# Public-release note

This curated notebook contains the original research workflow with execution outputs removed. Bloomberg and other licensed source data are not distributed. To run it, supply compatible files under `data/processed/` as described in `data/README.md`. Any results produced locally depend on the user's licensed data and are not included in this repository.


# Stationary Bootstrap Analysis

This notebook implements the stationary-bootstrap stage of the replication and produces strategy-level confidence intervals for risk-adjusted performance.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Locate the repository root when running from either the project or notebooks directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.bootstrap.stationary_bootstrap import (
    bootstrap_confidence_report,
    bootstrap_average_strategy_performance,
    strategy_differences,
    summarize_bootstrap_performance,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BOOTSTRAP_DIR = PROJECT_ROOT / "results" / "bootstrap"
BOOTSTRAP_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Main parameters: set RUN_FULL_BOOTSTRAP = True for the full replication
RUN_FULL_BOOTSTRAP = True

PAPER_N_SIMS = 1000
PAPER_PATHS_PER_SIM = 100
TEST_N_SIMS = 5
TEST_PATHS_PER_SIM = 10

N_SIMS = PAPER_N_SIMS if RUN_FULL_BOOTSTRAP else TEST_N_SIMS
PATHS_PER_SIM = PAPER_PATHS_PER_SIM if RUN_FULL_BOOTSTRAP else TEST_PATHS_PER_SIM
AVG_BLOCK_LENGTH = 2
RANDOM_STATE = 42  # reproducible bootstrap seed
BOOTSTRAP_METRICS = ("sharpe",)
SAVE_RAW_BOOTSTRAP = False

SAMPLE_START = "1982-01-31"
SAMPLE_END = "2011-12-31"
COUNTRIES = ["US", "UK", "DE"]
HORIZONS = [5, 10]
RUN_LABEL = "full" if RUN_FULL_BOOTSTRAP else "test"

pd.DataFrame([{
    "mode": RUN_LABEL,
    "n_sims": N_SIMS,
    "paths_per_sim": PATHS_PER_SIM,
    "avg_block_length": AVG_BLOCK_LENGTH,
    "metrics": ", ".join(BOOTSTRAP_METRICS),
}])

In [ ]:
# Load processed datasets and retain the paper's sample
datasets = {
    "US": pd.read_csv(PROCESSED_DIR / "us_data.csv", parse_dates=["Date"]),
    "UK": pd.read_csv(PROCESSED_DIR / "uk_data.csv", parse_dates=["Date"]),
    "DE": pd.read_csv(PROCESSED_DIR / "de_data.csv", parse_dates=["Date"]),
}

sample = {
    country: df[df["Date"].between(SAMPLE_START, SAMPLE_END)].reset_index(drop=True)
    for country, df in datasets.items()
}

pd.DataFrame(
    [{"Country": c, "Start": df["Date"].min(), "End": df["Date"].max(), "Rows": len(df)} for c, df in sample.items()]
)

In [ ]:
# Periodic strategies from Table 5: a zero threshold implies pure periodic rebalancing
strategies = {
    "Buy-and-hold": {"strategy": "buy_and_hold"},
    "Yearly rebalancing": {"strategy": "periodic", "frequency": "Y"},
    "Quarterly rebalancing": {"strategy": "periodic", "frequency": "Q"},
    "Monthly rebalancing": {"strategy": "periodic", "frequency": "M"},
}

comparison_pairs = {
    "M-BAH": ("Monthly rebalancing", "Buy-and-hold"),
    "Q-BAH": ("Quarterly rebalancing", "Buy-and-hold"),
    "Y-BAH": ("Yearly rebalancing", "Buy-and-hold"),
}

In [ ]:
# Generate bootstrap paths and evaluate strategies by country and horizon
perf = pd.concat(
    [
        bootstrap_average_strategy_performance(
            df,
            strategies,
            horizon,
            n_sims=N_SIMS,
            paths_per_sim=PATHS_PER_SIM,
            avg_block_length=AVG_BLOCK_LENGTH,
            random_state=RANDOM_STATE + i * 100 + horizon,
            required_cols=("Equity_Return", f"Bond_{horizon}Y_Return", "RF_Return"),
            metrics=BOOTSTRAP_METRICS,
        ).assign(Country=country)
        for i, (country, df) in enumerate(sample.items())
        for horizon in HORIZONS
    ],
    ignore_index=True,
)

perf.head()

In [ ]:
# Table 5: mean Sharpe ratio across bootstrap simulations
summary = pd.concat(
    [summarize_bootstrap_performance(perf[perf["Country"] == country]).assign(Country=country) for country in COUNTRIES],
    ignore_index=True,
)

strategy_order = ["Buy-and-hold", "Yearly rebalancing", "Quarterly rebalancing", "Monthly rebalancing"]
summary["Strategy"] = pd.Categorical(summary["Strategy"], strategy_order, ordered=True)
summary = summary.sort_values(["Horizon", "Strategy", "Country"]).reset_index(drop=True)

table5_sharpe = summary.pivot(index=["Horizon", "Strategy"], columns="Country", values="sharpe")[COUNTRIES].round(4)
table5_sharpe

In [ ]:
# Published Table 5 values used as a consistency check for the full replication
paper_table5 = pd.DataFrame(
    [
        (5, "Buy-and-hold", 0.554, 0.336, 0.315),
        (5, "Yearly rebalancing", 0.580, 0.354, 0.354),
        (5, "Quarterly rebalancing", 0.583, 0.356, 0.359),
        (5, "Monthly rebalancing", 0.580, 0.355, 0.356),
        (10, "Buy-and-hold", 0.552, 0.374, 0.314),
        (10, "Yearly rebalancing", 0.579, 0.389, 0.355),
        (10, "Quarterly rebalancing", 0.579, 0.390, 0.356),
        (10, "Monthly rebalancing", 0.575, 0.389, 0.351),
    ],
    columns=["Horizon", "Strategy", "Paper_US", "Paper_UK", "Paper_DE"],
)

table5_comparison = summary.pivot(index=["Horizon", "Strategy"], columns="Country", values="sharpe").reset_index()
table5_comparison = table5_comparison.merge(paper_table5, on=["Horizon", "Strategy"])
for country in COUNTRIES:
    table5_comparison[f"Diff_{country}"] = table5_comparison[country] - table5_comparison[f"Paper_{country}"]

table5_comparison.round(4)

In [ ]:
# Table 6 Panel A: Sharpe differences between periodic rebalancing and buy-and-hold
diffs = pd.concat(
    [
        strategy_differences(perf[perf["Country"] == country], comparison_pairs, metric="sharpe").assign(Country=country)
        for country in COUNTRIES
    ],
    ignore_index=True,
)

ci = pd.concat(
    [bootstrap_confidence_report(diffs[diffs["Country"] == country], metric="sharpe").assign(Country=country) for country in COUNTRIES],
    ignore_index=True,
)

ci["Interval"] = ci.apply(lambda r: f"{r['CI_Lower']:.4f} {r['CI_Upper']:.4f}{r['Significance']}", axis=1)
table6_panel_a = ci.pivot(index=["Horizon", "Comparison"], columns="Country", values="Interval")[COUNTRIES]
table6_panel_a

In [ ]:
# Save bootstrap results and summary tables
summary.to_csv(BOOTSTRAP_DIR / f"stationary_bootstrap_summary_{RUN_LABEL}.csv", index=False)
table5_sharpe.to_csv(BOOTSTRAP_DIR / f"stationary_bootstrap_table5_sharpe_{RUN_LABEL}.csv")
table5_comparison.to_csv(BOOTSTRAP_DIR / f"stationary_bootstrap_table5_comparison_{RUN_LABEL}.csv", index=False)
diffs.to_csv(BOOTSTRAP_DIR / f"stationary_bootstrap_sharpe_differences_{RUN_LABEL}.csv", index=False)
ci.to_csv(BOOTSTRAP_DIR / f"stationary_bootstrap_table6_panel_a_ci_{RUN_LABEL}.csv", index=False)
table6_panel_a.to_csv(BOOTSTRAP_DIR / f"stationary_bootstrap_table6_panel_a_{RUN_LABEL}.csv")

if SAVE_RAW_BOOTSTRAP:
    perf.to_csv(BOOTSTRAP_DIR / f"stationary_bootstrap_raw_performance_{RUN_LABEL}.csv", index=False)

with pd.ExcelWriter(BOOTSTRAP_DIR / f"stationary_bootstrap_results_{RUN_LABEL}.xlsx", engine="openpyxl") as writer:
    table5_sharpe.to_excel(writer, sheet_name="table5_sharpe")
    table5_comparison.to_excel(writer, sheet_name="table5_vs_paper", index=False)
    table6_panel_a.to_excel(writer, sheet_name="table6_panel_a")
    ci.to_excel(writer, sheet_name="ci_details", index=False)
    summary.to_excel(writer, sheet_name="summary", index=False)

BOOTSTRAP_DIR / f"stationary_bootstrap_results_{RUN_LABEL}.xlsx"